In [ ]:
import os
import shlex
import subprocess

WORKSHOP_RESOURCE_GROUP = os.getenv("WORKSHOP_RESOURCE_GROUP", "rg-delete-me-01").strip()
WORKSHOP_AUTH_MODE = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
print(f"Notebook config: WORKSHOP_RESOURCE_GROUP={WORKSHOP_RESOURCE_GROUP}, WORKSHOP_AUTH_MODE={WORKSHOP_AUTH_MODE}")

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    WORKSHOP_RESOURCE_GROUP,
    "--auth-mode",
    WORKSHOP_AUTH_MODE,
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

In [ ]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

# Workshop 1: Foundry Model Deployments

This notebook mirrors docs/foundry.md and deploys both a chat model and an embedding model with Azure CLI.

Deployment zone options:
- global (maps to GlobalStandard)
- data_zone (maps to DataZoneStandard)

Set MODEL_DEPLOYMENT_ZONE before running Cell 1 if you want data-zone deployments.

In [ ]:
from workshop_bootstrap import (
    WorkshopConstants,
    ensure_cognitiveservices_extension,
    get_available_models,
    run_az,
)

if not config.foundry_account_name:
    raise ValueError("FOUNDRY_ACCOUNT_NAME could not be resolved. Set it in CONFIG_OVERRIDES or environment.")

ensure_cognitiveservices_extension()
available_models = get_available_models(config)
preferred_sku = WorkshopConstants.MODEL_SKU_BY_ZONE[config.model_zone]

print(f"Resolved model zone: {config.model_zone}")
print(f"Preferred deployment SKU: {preferred_sku}")

matching = [
    {
        "name": model.get("name", ""),
        "format": model.get("format", ""),
        "version": str(model.get("version", "")),
        "sku": (model.get("skus") or [{}])[0].get("name", ""),
        "capacity": (model.get("skus") or [{}])[0].get("capacity", {}).get("default", ""),
    }
    for model in available_models
    if model.get("name") in set(WorkshopConstants.CHAT_MODEL_CANDIDATES + WorkshopConstants.EMBEDDING_MODEL_CANDIDATES)
]

print("Candidate models available in this account:")
for row in matching:
    print(row)

In [ ]:
from workshop_bootstrap import ensure_models_deployed

deployment_summary = ensure_models_deployed(config)
print("Deployment summary:")
for key, value in deployment_summary.items():
    print(f"- {key}: {value}")

CHAT_DEPLOYMENT_NAME = deployment_summary["chat_deployment_name"]
EMBEDDING_DEPLOYMENT_NAME = deployment_summary["embedding_deployment_name"]
CHAT_MODEL_NAME = deployment_summary["chat_model_name"]
EMBEDDING_MODEL_NAME = deployment_summary["embedding_model_name"]

In [ ]:
import os
import subprocess

from workshop_bootstrap import list_deployments

account_details = run_az([
    "cognitiveservices",
    "account",
    "show",
    "--name",
    config.foundry_account_name,
    "--resource-group",
    config.resource_group_name,
], expect_json=True)

inference_endpoint = account_details.get("properties", {}).get("endpoints", {}).get("Azure AI Model Inference API", "")
print("Inference endpoint:", inference_endpoint)

auth_mode = os.getenv("WORKSHOP_AUTH_MODE", "key").strip().lower()
keys = {}
if auth_mode == "key":
    try:
        keys = run_az([
            "cognitiveservices",
            "account",
            "keys",
            "list",
            "--name",
            config.foundry_account_name,
            "--resource-group",
            config.resource_group_name,
        ], expect_json=True)
    except subprocess.CalledProcessError as exc:
        message = (exc.stderr or str(exc)).strip()
        print(f"Primary key lookup skipped: {message}")
else:
    print("Managed identity mode detected: skipping key retrieval.")

print("Primary key loaded:", bool(keys.get("key1")))

print("\nCurrent deployments:")
for deployment in list_deployments(config):
    print(f"- {deployment.get('name', '')}: {deployment.get('properties', {}).get('provisioningState', '')}")

In [ ]:
import json
import os

values = {
    "CHAT_DEPLOYMENT_NAME": str(globals().get("CHAT_DEPLOYMENT_NAME", os.getenv("CHAT_DEPLOYMENT_NAME", ""))).strip(),
    "CHAT_MODEL_NAME": str(globals().get("CHAT_MODEL_NAME", os.getenv("CHAT_MODEL_NAME", ""))).strip(),
    "EMBEDDING_DEPLOYMENT_NAME": str(globals().get("EMBEDDING_DEPLOYMENT_NAME", os.getenv("EMBEDDING_DEPLOYMENT_NAME", ""))).strip(),
    "EMBEDDING_MODEL_NAME": str(globals().get("EMBEDDING_MODEL_NAME", os.getenv("EMBEDDING_MODEL_NAME", ""))).strip(),
}
print("# Copy and paste the following into your next notebook to set the environment variables for the current kernel:")
print("import os")
for key in [
    "CHAT_DEPLOYMENT_NAME",
    "CHAT_MODEL_NAME",
    "EMBEDDING_DEPLOYMENT_NAME",
    "EMBEDDING_MODEL_NAME",
]:
    print(f'os.environ["{key}"] = {json.dumps(values[key])}')